## Entrenamiento y comparacion de modelos

### Imports

In [1]:

import torch
import time
from ultralytics import YOLO
from ultralytics.utils.benchmarks import benchmark
import pandas as pd
import sys
from pathlib import Path
import gc
import yaml
import easyocr
import os
import editdistance
import cv2
import re

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))
    
from config import YAML_PATH, CUSTOM_MODEL_WEIGHTS_PATH, OCR_MODEL_WEIGHTS_PATH
from notebook_utils import EasyOCRModel, cargar_texto_real_desde_txt

In [ ]:
model = "yolov8n.pt"
epochs = 50

### Entrenamiento

In [ ]:
def train_model(model, data_path, epochs):
  model = YOLO(model)
  results = model.train(data=data_path,
                        epochs=epochs,
                        imgsz=640,
                        device='cuda' if torch.cuda.is_available() else 'cpu')

train_model(model, YAML_PATH, epochs)

### Evaluacion y comparacion | Deteccion de matriculas

In [ ]:
def benchmark_comparativo(model_path, data_path):
    print(f"\n>>> Analizando métricas detalladas: {model_path}")
    
    #Limpiar caché de GPU antes de cada evaluación
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    model = YOLO(model_path)
    
    #Warmup
    dummy_input = torch.randn(1, 3, 640, 640).to("cuda")
    for _ in range(10): _ = model(dummy_input, verbose=False)
    
    #Evaluacion
    metrics = model.val(data=data_path, imgsz=640, verbose=False)
    
    #Uso de VRAM
    peak_vram = torch.cuda.max_memory_allocated() / 1024**2
    
    return {
        "Modelo": model_path,
        "Precisión (P)": round(metrics.box.mp, 4),
        "Recall (R)": round(metrics.box.mr, 4),
        "F1-Score": round(metrics.box.f1.max(), 4),
        "mAP@50": round(metrics.box.map50, 4),
        "mAP@50-95": round(metrics.box.map, 4),
        "Latencia (ms)": round(metrics.speed['inference'], 2),
        "VRAM (MB)": round(peak_vram, 2)
    }

#Agregar todos los modelos re-entrandos para generar la tabla comparativa
modelos = [CUSTOM_MODEL_WEIGHTS_PATH]
resumen = []

for m in modelos:
    resumen.append(benchmark_comparativo(m, YAML_PATH))

df = pd.DataFrame(resumen)
df.to_csv("comparativa_modelos_optuna.csv", index=False)
print("\n--- TABLA COMPARATIVA ---")
print(df.to_markdown(index=False))

### Evaluacion y comparacion | OCR

In [ ]:
def benchmark_comparativo_ocr(model_path, data_path, use_easyocr=False):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    with open(data_path, 'r') as f:
        data_config = yaml.safe_load(f)
    base_dir = os.path.dirname(os.path.abspath(data_path))
    val_images_path = os.path.join(base_dir, data_config['val'])
    
    # Necesitamos los nombres de las clases del modelo YOLO para leer los .txt
    temp_model = YOLO(CUSTOM_MODEL_WEIGHTS_PATH)
    model_names = temp_model.names

    total_cer, placas_perfectas, total_placas, total_latencia = 0, 0, 0, 0
    
    if not use_easyocr:
        #Evaluamos YOLO
        model = YOLO(model_path)
        val_res = model.val(data=data_path, imgsz=640, verbose=False)
        mAP = round(val_res.box.map50, 4)
        results = model.predict(val_images_path, imgsz=640, verbose=False)
        
        for res in results:
            if len(res.boxes) > 0:
                clases_p = res.boxes.cls.cpu().numpy()
                x_coords_p = res.boxes.xywh[:, 0].cpu().numpy()
                sorted_indices = x_coords_p.argsort()
                pred_text = "".join([model_names[int(clases_p[i])] for i in sorted_indices])
                
                true_text = cargar_texto_real_desde_txt(res.path, model_names)
                if true_text:
                    dist = editdistance.eval(pred_text, true_text)
                    total_cer += dist / max(len(true_text), 1)
                    if dist == 0: placas_perfectas += 1
                    total_placas += 1
        latencia = round(val_res.speed['inference'], 2)
    else:
        #Evaluamos EasyOCR
        e_ocr = EasyOCRModel()
        mAP = "N/A"
        img_files = [os.path.join(val_images_path, f) for f in os.listdir(val_images_path) 
                     if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        
        for img_p in img_files:
            pred_text, lat = e_ocr.predict_ocr(img_p)
            true_text = cargar_texto_real_desde_txt(img_p, model_names)
            if true_text:
                dist = editdistance.eval(pred_text, true_text)
                total_cer += dist / max(len(true_text), 1)
                if dist == 0: placas_perfectas += 1
                total_placas += 1
                total_latencia += lat
        latencia = round(total_latencia / len(img_files), 2) if img_files else 0

    return {
        "Modelo": "EasyOCR" if use_easyocr else os.path.basename(model_path),
        "mAP@50": mAP,
        "CER (Real)": round(total_cer / total_placas, 4) if total_placas > 0 else 1,
        "Exactitud Placa": f"{round((placas_perfectas / total_placas) * 100, 2) if total_placas > 0 else 0}%",
        "Latencia (ms)": latencia,
        "VRAM (MB)": round(torch.cuda.max_memory_allocated() / 1024**2, 2) if torch.cuda.is_available() else 0
    }


resumen = []
# 1. Yolo8n personalizado
resumen.append(benchmark_comparativo_ocr(OCR_MODEL_WEIGHTS_PATH, YAML_PATH, use_easyocr=False))

#Añadir modelos propios para comparar
# ...

# 2. EasyOCR
resumen.append(benchmark_comparativo_ocr(None, YAML_PATH, use_easyocr=True))

df = pd.DataFrame(resumen)
df.to_csv("comparativa_ocr_alpr.csv", index=False)
print("\n--- TABLA COMPARATIVA FINAL ---")
print(df.to_markdown(index=False))